## Environmental Context

This notebook fetches gridded environmental data from [Bio-Oracle](https://erddap.bio-oracle.org/erddap/griddap) and seafloor lithology from the [NOAA/NCEI IMLGS](https://www.ncei.noaa.gov/erddap/tabledap/IMLGS_Marine_Geology.html) for the spatial extent of the coral dataset. Bio-Oracle layers use depth-mean baseline climatologies (2000–2019 or 2000–2018 depending on the variable). Each section fetches one variable, joins it onto the cleaned coral records by nearest grid cell or sample point, and spot-checks coverage where in-situ measurements are available.

Variables fetched:

- **Temperature** (`thetao_mean`) — Bio-Oracle, baseline 2000–2019
- **Salinity** (`so_mean`) — Bio-Oracle, baseline 2000–2019
- **Dissolved oxygen** (`o2_mean`) — Bio-Oracle, baseline 2000–2018
- **pH** (`ph_mean`) — Bio-Oracle, baseline 2000–2018
- **Nitrate** (`no3_mean`) — Bio-Oracle, baseline 2000–2018
- **Phosphate** (`po4_mean`) — Bio-Oracle, baseline 2000–2018
- **Chlorophyll-a** (`chl_mean`) — Bio-Oracle, baseline 2000–2018
- **Current speed** (`sws_mean`) — Bio-Oracle, baseline 2000–2019
- **Seafloor lithology** (`lith1`) — IMLGS primary lithology classification


In [5]:
from collections.abc import Collection

import pandas as pd

from se_deep_coral_communities.data_loading import (
    BioOracleChlorophyllSource,
    BioOracleCurrentSpeedSource,
    BioOracleNitrateSource,
    BioOracleOxygenSource,
    BioOraclePhosphateSource,
    BioOraclePHSource,
    BioOracleSalinitySource,
    BioOracleTemperatureSource,
    IMLGSSubstrateSource,
)

In [6]:
df = pd.read_parquet("../data/processed/se_coral_records_cleaned.parquet")

In [7]:
def print_coverage(df: pd.DataFrame, columns: Collection[str]):
    print(f"Rows: {len(df)}")
    print("Coverage:")
    for col in columns:
        print(f"  {col}: {df[col].notna().mean():.1%}")

## Temperature

`thetao_mean` tracks in-situ temperature closely (median difference ~ 1 °C). The gridded value fills gaps where no in-situ measurement was recorded.


In [8]:
df = BioOracleTemperatureSource().fetch_join(df)

print_coverage(df, ["thetao_mean"])

Rows: 71499
Coverage:
  thetao_mean: 100.0%


## Salinity


In [9]:
df = BioOracleSalinitySource().fetch_join(df)

print_coverage(df, ["so_mean"])

Rows: 71499
Coverage:
  so_mean: 100.0%


## Dissolved oxygen


In [10]:
df = BioOracleOxygenSource().fetch_join(df)

print_coverage(df, ["o2_mean"])

Rows: 71499
Coverage:
  o2_mean: 100.0%


## pH


In [11]:
df = BioOraclePHSource().fetch_join(df)

print_coverage(df, ["ph_mean"])

Rows: 71499
Coverage:
  ph_mean: 100.0%


## Seafloor lithology

`lith1` is the primary lithology classification from the nearest NOAA/NCEI IMLGS geological sample with water depth ≥ 50 m. Unlike the Bio-Oracle layers, this is a categorical variable assigned by nearest-neighbour join to physical dredge and core samples. Distance to the nearest sample varies; treat assignments for isolated events cautiously.


In [12]:
df = IMLGSSubstrateSource().fetch_join(df)

print_coverage(df, ["lith1"])

Rows: 71499
Coverage:
  lith1: 100.0%


## Nitrate

`no3_mean` is the depth-mean average dissolved nitrate concentration (mol m⁻³) from Bio-Oracle, baseline 2000–2018.


In [13]:
df = BioOracleNitrateSource().fetch_join(df)

print_coverage(df, ["no3_mean"])

Rows: 71499
Coverage:
  no3_mean: 100.0%


## Phosphate

`po4_mean` is the depth-mean average dissolved phosphate concentration (mol m⁻³) from Bio-Oracle, baseline 2000–2018.


In [14]:
df = BioOraclePhosphateSource().fetch_join(df)

print_coverage(df, ["po4_mean"])

Rows: 71499
Coverage:
  po4_mean: 100.0%


## Chlorophyll-a

`chl_mean` is the depth-mean average chlorophyll-a concentration (mg m⁻³) from Bio-Oracle, baseline 2000–2018.


In [15]:
df = BioOracleChlorophyllSource().fetch_join(df)

print_coverage(df, ["chl_mean"])

Rows: 71499
Coverage:
  chl_mean: 100.0%


## Current speed

`sws_mean` is the depth-mean average sea-water speed (m s⁻¹) from Bio-Oracle, baseline 2000–2019.


In [16]:
df = BioOracleCurrentSpeedSource().fetch_join(df)

print_coverage(df, ["sws_mean"])

Rows: 71499
Coverage:
  sws_mean: 100.0%


## Saving Results with Added Context


In [20]:
df.sample(10, random_state=29)

,lat,lon,catalog_num,record_type,loc_accuracy,scientific_nm,depth,gis_depth,large_marine_ecosystem,country,...,is_shallow,thetao_mean,so_mean,o2_mean,ph_mean,lith1,no3_mean,po4_mean,chl_mean,sws_mean
69134,29.32356,-87.77881,1682116,video observation,20m,Nicella,102.0,127.0,Gulf of Mexico,USA,...,False,18.237656,36.364597,133.062347,7.942151,terrigenous,13.107030,0.870360,0.075534,0.071713
52125,29.43453,-87.56722,1664837,video observation,20m,Serranus notospilus,82.0,80.0,Gulf of Mexico,USA,...,False,19.794005,36.403882,176.205857,8.034090,terrigenous,5.836460,0.414622,0.221213,0.092894
63559,29.35639,-87.73631,1676375,video observation,20m,Antipathes furcata,98.0,106.0,Gulf of Mexico,USA,...,False,19.859943,36.411068,162.956164,8.009709,terrigenous,7.723300,0.531825,0.174480,0.069814
34048,29.58529,-87.34873,1643327,video observation,20m,Ellisella,99.0,100.0,Gulf of Mexico,USA,...,False,16.937008,36.222962,124.207244,7.919049,"calcareous, foraminifera",15.413552,1.035322,0.039368,0.069497
50022,29.44957,-87.59139,1661533,video observation,20m,Malacalcyonacea,62.0,72.0,Gulf of Mexico,USA,...,False,19.794005,36.403882,176.205857,8.034090,terrigenous,5.836460,0.414622,0.221213,0.092894
71316,29.32468,-87.77667,1684332,video observation,20m,Scorpaenidae,102.0,127.0,Gulf of Mexico,USA,...,False,18.237656,36.364597,133.062347,7.942151,terrigenous,13.107030,0.870360,0.075534,0.071713
5040,32.09338,-77.16210,969513,video observation,20m,Primnoidae,893.0,893.0,Southeast U.S. Continental Shelf,USA,...,False,8.506715,35.207379,213.595420,7.961154,"calcareous, foraminifera",20.079743,1.302787,0.003869,0.255259
39007,29.44385,-87.54603,1648470,video observation,20m,Antipathes furcata,76.0,74.0,Gulf of Mexico,USA,...,False,18.177869,36.341822,140.885640,7.963790,terrigenous,11.399956,0.764854,0.106624,0.079982
45640,29.43955,-87.58395,1656092,video observation,20m,Stichopathes,68.0,76.0,Gulf of Mexico,USA,...,False,19.794005,36.403882,176.205857,8.034090,terrigenous,5.836460,0.414622,0.221213,0.092894
37828,29.41938,-87.56252,1647219,video observation,20m,Pennatuloidea,92.0,97.0,Gulf of Mexico,USA,...,False,19.794005,36.403882,176.205857,8.034090,terrigenous,5.836460,0.414622,0.221213,0.092894


In [18]:
out_path = "../data/processed/bio_oracle_context.parquet"
df.to_parquet(out_path, index=False)
print(f"Saved {len(df):,} rows to {out_path}")

Saved 71,499 rows to ../data/processed/bio_oracle_context.parquet
